# Clase 2 — Las particularidades del dato espacial

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 2 — Particularidades de la información geográfica |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Cambia la respuesta si cambio la grilla?

En la clase pasada calculamos escuelas por habitante **por provincia**, y descubrimos que
el conteo absoluto y la tasa daban mapas opuestos. Hoy vamos más al fondo: vamos a ver que
**la propia elección de la unidad territorial modifica el resultado**, y que esa elección
suele tomarse por conveniencia sin que nadie la
justifique.

Esto no es una curiosidad técnica. Si el máximo de un indicador cambia según dónde
dibujemos las líneas de una grilla, entonces frases como *"la zona más crítica del
aglomerado"* dependen de una decisión metodológica que el mapa no muestra.

Los datos espaciales tienen un conjunto de propiedades que los vuelven distintos de una
tabla común, y que **rompen supuestos** de la estadística que solemos dar por sentados.
Hoy enunciaremos y trabajaremos con algunas de estas propriedades.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Leer** una capa desde el repositorio del curso con `gpd.read_file()`, y **explicar**
   qué pierde el dato según el formato en que esté guardado.
2. **Explicar** la primera ley de la geografía y por qué amenaza el supuesto de
   independencia de las observaciones.
3. **Demostrar** que la longitud de un objeto geográfico depende de la escala a la que se lo mida.
4. **Distinguir** los dos componentes del problema de la unidad de área modificable:
   el efecto de agregación y el de zonificación.
5. **Cuantificar** el efecto de borde en una zona de estudio.
6. **Reconocer** cuándo un valor agregado por área representa mal a la población que contiene.

## 3. Material de esta clase

- **Presentación Clase 1**, diapositivas 17–22.
- **Presentación Clase 2**, diapositivas 1-15.


| Bloque de la notebook | Diapositivas |
|---|---|
| Formatos de archivo | 17–22 |
| Primera ley de la geografía y autocorrelación | 12 |
| Escala de análisis | 10 |
| Unidad de área modificable | 11 |
| Efecto de borde | 14 |
| Localización representada | 15 |

📖 Olaya, V. *Sistemas de Información Geográfica*, capítulo 11.

## 4. Preparación del entorno

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "folium==0.17.0" "matplotlib==3.9.2"
!wget -q -O sig_utils.py https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/sig_utils.py

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

from sig_utils import grilla, chequear_crs, CRS_ARGENTINA

# Los datos del curso viven en un repositorio público de GitHub.
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

# Sistemas de referencia que vamos a usar hoy.
EQUIVALENTE  = CRS_ARGENTINA["sudamerica_equivalente"]   # para áreas
EQUIDISTANTE = CRS_ARGENTINA["sudamerica_equidistante"]  # para distancias

print(f"GeoPandas {gpd.__version__} — entorno listo")

In [ ]:
# read_file() abre un archivo geoespacial. La ruta puede ser local o una dirección web.
provincias = gpd.read_file(DATOS + "provincias_arg.gpkg")
escuelas   = gpd.read_file(DATOS + "escuelas_primarias.gpkg")
ruta40     = gpd.read_file(DATOS + "ruta_nacional_40.gpkg")

for nombre, capa in [("provincias", provincias), ("escuelas", escuelas), ("ruta 40", ruta40)]:
    print(f"{nombre:12} {len(capa):>6} filas   CRS: {capa.crs.to_string()}")

---

## 5. Recapitulación: dónde quedamos

En la Clase 1 construimos un indicador de escuelas por cada 10.000 habitantes a nivel
provincial y vimos que el conteo absoluto y la tasa producían mapas opuestos.

Reconstruimos ese indicador en una celda, porque lo vamos a usar todo el día.

In [ ]:
# Cuántas escuelas tiene cada provincia
conteo = escuelas.groupby("provincia").size().reset_index(name="escuelas")

# Se lo pegamos a la capa de provincias, que ya trae la población
tasa_provincial = provincias.merge(conteo, on="provincia", how="left")
tasa_provincial["tasa"] = tasa_provincial["escuelas"] / tasa_provincial["poblacion"] * 10_000

print("Provincias:", len(tasa_provincial))
print("Tasa mínima:", round(tasa_provincial["tasa"].min(), 1), "escuelas cada 10.000 habitantes")
print("Tasa máxima:", round(tasa_provincial["tasa"].max(), 1), "escuelas cada 10.000 habitantes")

---

## 6. Formatos de archivo: qué le hace el envase al dato

📽️ *Presentación Clase 1, diapositivas 17–22.*

### 🧭 Concepto

Las tres capas que acabamos de leer eran GeoPackage, y las leímos con una sola línea:
`gpd.read_file(ruta)`. Esa misma función abre casi cualquier formato geoespacial, así que
el formato parece un detalle administrativo. No lo es: **cada formato garantiza cosas
distintas, y lo que no garantiza lo pierde en silencio.**

| Formato | Archivos | Fortaleza | Límite importante |
|---|---|---|---|
| **GeoPackage** (`.gpkg`) | 1 | Abierto, un solo archivo, varias capas, nombres de campo largos | Menos difundido de lo que merece |
| **GeoJSON** (`.geojson`) | 1 | Texto plano, legible a ojo, nativo de la web | Pesado; en la práctica, solo EPSG:4326 |
| **Shapefile** (`.shp`) | **4 como mínimo** | Estándar de facto: todo programa lo abre | Nombres de campo de **10 caracteres**; codificación no declarada |
| **CSV con coordenadas** | 1 | Sale de cualquier sistema que no sea geográfico | **No guarda el CRS** |

En la Clase 1 leímos esa tabla y confiamos en ella. Hoy la vamos a abrir: cada una de las
tres primeras filas es un archivo real, en el repositorio del curso, que podemos mirar por
fuera de Python y romper a propósito.

### 👀 Andá a mirar los archivos

Abrí esta carpeta del repositorio del curso en otra pestaña del navegador:

🔗 **https://github.com/renzoepolo/sig-ciencias-sociales/tree/main/datos/formatos**

Ahí están **las mismas 24 provincias tres veces**, una por formato. Mirá la lista antes de
seguir y contestá dos cosas:

### 👀 Un Shapefile no es *un* archivo

Es lo primero que llama la atención de esa lista. Vamos a comprobarlo de la peor manera
posible: bajando solamente el archivo que le da nombre al formato, el `.shp`, y pidiéndole
a GeoPandas que lo abra.

In [ ]:
import os
import urllib.request

FORMATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/formatos/"

# Bajamos SOLO el .shp
urllib.request.urlretrieve(FORMATOS + "provincias.shp", "provincias.shp")

try:
    gpd.read_file("provincias.shp")
except Exception as error:
    print("GeoPandas no pudo abrirlo:")
    print(error)

El `.shp` guarda las geometrías y nada más. El resto de la capa está repartido en otros
archivos, y **ninguno es opcional**:

| Archivo | Qué guarda |
|---|---|
| `.shp` | Las geometrías: los puntos, las líneas o los polígonos |
| `.shx` | Un índice que dice en qué posición del `.shp` empieza cada geometría |
| `.dbf` | La tabla de atributos: todo lo que no es geometría |
| `.prj` | El sistema de coordenadas |
| `.cpg` | La codificación de caracteres del `.dbf` — **este no está**, y más abajo se va a notar |

Si copiás una capa de una carpeta a otra y te olvidás uno, no te quedás con una capa
incompleta: no te quedás con nada. Bajemos los tres que faltan.

In [ ]:
for extension in ["shx", "dbf", "prj"]:
    archivo = "provincias." + extension
    urllib.request.urlretrieve(FORMATOS + archivo, archivo)

# Cuánto ocupa cada pieza del conjunto
for extension in ["shp", "shx", "dbf", "prj"]:
    archivo = "provincias." + extension
    print(f"{archivo:18} {os.path.getsize(archivo) / 1024:>9.1f} KB")

### 👀 Tres formatos, el mismo mapa

Ahora sí, las tres capas, leídas las tres con **la misma función**. Bajamos también el
GeoPackage y el GeoJSON, para poder compararlos como archivos más adelante.

In [ ]:
for archivo in ["provincias.gpkg", "provincias.geojson"]:
    urllib.request.urlretrieve(FORMATOS + archivo, archivo)

prov_shp     = gpd.read_file("provincias.shp")
prov_gpkg    = gpd.read_file("provincias.gpkg")
prov_geojson = gpd.read_file("provincias.geojson")

capas = [("Shapefile", prov_shp), ("GeoPackage", prov_gpkg), ("GeoJSON", prov_geojson)]

fig, ejes = plt.subplots(1, 3, figsize=(13, 6))
for eje, (nombre, capa) in zip(ejes, capas):
    capa.plot(ax=eje, color="#8da0cb", edgecolor="white", linewidth=0.4)
    eje.set_title(f"{nombre}\n{len(capa)} provincias")
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### ▶️ El mapa es el mismo. La tabla, no.

Los tres mapas son indistinguibles: las geometrías atravesaron los tres formatos sin
perder nada. Miremos ahora los nombres de las columnas, uno al lado del otro.

In [ ]:
for columna_gpkg, columna_shp in zip(prov_gpkg.columns, prov_shp.columns):
    aviso = "  <-- cambió" if columna_gpkg != columna_shp else ""
    print(f"{columna_gpkg:20} {columna_shp:12}{aviso}")

In [ ]:
# Y los nombres de las provincias, que llevan acentos
print("GeoPackage:", prov_gpkg["provincia"].tolist()[:4])
print("Shapefile: ", prov_shp["provincia"].tolist()[:4])

# El mismo archivo, diciéndole a mano en qué codificación está escrito
prov_shp_utf8 = gpd.read_file("provincias.shp", encoding="utf-8")
print("Shapefile: ", prov_shp_utf8["provincia"].tolist()[:4], "  <-- con encoding='utf-8'")

### ✅ Comprobación — ¿es el mismo dato o no?

Antes de sacar conclusiones: si los tres archivos tienen las mismas 24 filas y la misma
población total, entonces el dato es el mismo y lo único que cambió fue el envase.

In [ ]:
print(f"{'formato':12} {'filas':>6} {'población total':>18}")
print(f"{'Shapefile':12} {len(prov_shp):>6} {prov_shp['poblacion_'].sum():>18,}")
print(f"{'GeoPackage':12} {len(prov_gpkg):>6} {prov_gpkg['poblacion_total'].sum():>18,}")
print(f"{'GeoJSON':12} {len(prov_geojson):>6} {prov_geojson['poblacion_total'].sum():>18,}")

# Y cuánto pesa ese mismo dato en cada formato
peso_shapefile = 0
for extension in ["shp", "shx", "dbf", "prj"]:
    peso_shapefile += os.path.getsize("provincias." + extension)

print()
print(f"{'Shapefile (4 archivos)':24} {peso_shapefile / 1024 / 1024:>6.1f} MB")
print(f"{'GeoPackage':24} {os.path.getsize('provincias.gpkg') / 1024 / 1024:>6.1f} MB")
print(f"{'GeoJSON':24} {os.path.getsize('provincias.geojson') / 1024 / 1024:>6.1f} MB")

### 🔍 Interpretación

El dato es el mismo: 24 filas y 45.617.341 habitantes en los tres archivos. Sin embargo,
para sumar esa misma columna en el Shapefile hubo que escribir `poblacion_` en lugar de
`poblacion_total`. Pasaron tres cosas, y ninguna avisó.

**1. Los nombres se cortaron a diez caracteres.** `total_hogares` quedó como `total_hoga`,
que no significa nada: alguien tiene que acordarse de qué era. Y `provincia_id` quedó como
`provincia_`, a un carácter de distancia de `provincia`, que es otra columna del mismo
archivo.

**2. Dos columnas distintas quisieron llamarse igual.** `poblacion_total` y
`poblacion_por_km2` comparten sus primeros diez caracteres. Como no puede haber dos
columnas con el mismo nombre, el programa que escribió el archivo desempató por su cuenta:
`poblacion_` y `poblacio_1`. Una es una población y la otra una densidad, y lo único que
las distingue es un dígito. Nada dentro del archivo dice cuál era cuál.

**3. Los acentos se rompieron.** "Córdoba" se lee "CÃ³rdoba". El texto del `.dbf` está
escrito en UTF-8, pero al conjunto le falta el `.cpg` que lo declare, así que el programa
que lo abre tiene que adivinar la codificación —y adivina otra—. Se arregla pasando
`encoding="utf-8"`, como en la celda anterior, pero **eso exige saber de antemano cuál era
la codificación original**: el archivo no lo dice, y si uno se equivoca no salta ningún
error, solo aparecen nombres raros.

Las tres son pérdidas del envase, no del dato: el GeoPackage y el GeoJSON, con exactamente
la misma información adentro, no tienen ninguna. A cambio, el Shapefile es el más liviano
de los tres y el GeoJSON pesa casi el doble, porque guarda cada coordenada como texto.

> ⚠️ **Esto no es una curiosidad histórica.** El GeoPackage de provincias que venimos
> usando —`poblacion`, `total_hogares`, "Córdoba"— salió de un Shapefile del IGN que traía
> `poblacion_`, `total_hoga` y "CÃ³rdoba". Esos nombres hubo que reconstruirlos a mano,
> preguntándole a la documentación de la fuente qué había significado cada uno. Por eso en
> este curso, cuando haya que guardar una capa, la guardamos en `.gpkg`.

---

## 7. La primera ley de la geografía

📽️ *Presentación Clase 2, diapositiva 12.*

### 🧭 Concepto

Waldo Tobler la enunció en 1970, y es engañosamente simple:

> *"Todo está relacionado con todo lo demás, pero las cosas cercanas están más
> relacionadas entre sí que las lejanas."*

Suena a obviedad. No lo es: si fuera cierta, entonces **dos observaciones vecinas no son
dos observaciones independientes**. Y la independencia de las observaciones es un supuesto
de casi toda la estadística que usamos: de la regresión lineal, de los tests de
significancia, de los intervalos de confianza.

Dicho de otro modo: si los datos espaciales cumplen la ley de Tobler —y en general la
cumplen— entonces al aplicarles estadística convencional estamos **contando la misma
información más de una vez**.

Vamos a comprobar si se cumple, sin ninguna fórmula: comparando cuánto se diferencian las
provincias **vecinas** contra cuánto se diferencian provincias **tomadas al azar**.

### 👀 Primero, mirar el mapa

Antes de calcular nada, pintemos la tasa provincial que acabamos de reconstruir y
preguntémonos algo muy simple: **¿los colores parecidos están cerca unos de otros, o
aparecen desperdigados por el mapa?**

In [ ]:
fig, eje = plt.subplots(figsize=(6, 9))

tasa_provincial.plot(column="tasa", cmap="YlGnBu", ax=eje,
                     edgecolor="white", linewidth=0.5,
                     legend=True, legend_kwds={"shrink": 0.5,
                                               "label": "Escuelas cada 10.000 habitantes"})

eje.set_title("¿Los colores parecidos están cerca\nunos de otros?")
eje.set_axis_off()
plt.show()

### ▶️ Ahora, medirlo

Lo que se ve en el mapa hay que poder decirlo con un número. La forma más simple: comparar
cuánto se diferencian las provincias **vecinas** contra cuánto se diferencian **dos
provincias cualesquiera**. Si Tobler tiene razón, la primera diferencia tiene que ser menor.

In [ ]:
# 1 - Diferencia de tasa entre provincias que COMPARTEN FRONTERA
diferencias_vecinas = []

for geometria, tasa in zip(tasa_provincial["geometry"], tasa_provincial["tasa"]):
    vecinas = tasa_provincial[tasa_provincial.touches(geometria)] # Predicado geométrico: Tocar
    for tasa_vecina in vecinas["tasa"]:
        diferencias_vecinas.append(abs(tasa - tasa_vecina))

# 2 - Diferencia de tasa entre TODOS los pares de provincias, vecinas o no
tasas = list(tasa_provincial["tasa"])
diferencias_todas = []

for i in range(len(tasas)):
    for j in range(i + 1, len(tasas)):
        diferencias_todas.append(abs(tasas[i] - tasas[j]))

print("Pares de provincias vecinas:", len(diferencias_vecinas))
print("Pares de provincias en total:", len(diferencias_todas))

### ✅ Comprobación

In [ ]:
media_vecinas = sum(diferencias_vecinas) / len(diferencias_vecinas)
media_todas   = sum(diferencias_todas) / len(diferencias_todas)

print(f"Diferencia media entre provincias VECINAS:      {media_vecinas:.2f}")
print(f"Diferencia media entre DOS PROVINCIAS CUALESQUIERA: {media_todas:.2f}")

parecido = 100 * (1 - media_vecinas / media_todas)
print(f"\nLas vecinas se parecen un {parecido:.0f}% más que dos provincias cualesquiera")

assert media_vecinas < media_todas, "Si esto falla, no hay autocorrelación positiva"

### 🔍 Interpretación

Las provincias que comparten frontera se diferencian, en promedio, **2,33 puntos** de tasa;
dos provincias cualesquiera, **2,99**. Las vecinas se parecen un **22 % más** de lo que se
parecen dos provincias tomadas sin mirar dónde están. La ley de Tobler se cumple en estos
datos.

Eso tiene una consecuencia incómoda: si mañana ajustáramos una regresión con estas 24
provincias como observaciones independientes, estaríamos exagerando cuánta información
tenemos. Parte de lo que aporta cada provincia ya está aportado por sus vecinas, de modo
que los valores p saldrían más chicos de lo que corresponde y concluiríamos que hay
efectos significativos donde tal vez no los haya.

> ⚠️ **Ojo con la interpretación.** Que las provincias vecinas se parezcan **no** significa
> que una influya sobre la otra. Puede ser que compartan clima, historia productiva o
> composición demográfica. La autocorrelación describe un patrón; no identifica su causa.
> Vamos a medirla formalmente en la Clase 7, y ahí insistiremos con esto.

---

## 8. Escala: ¿cuánto mide la Ruta 40?

📽️ *Presentación Clase 2, diapositiva 10.*

### 🧭 Concepto

Parece una pregunta con una sola respuesta. No la tiene.

Cuando representamos un objeto sinuoso —una ruta, una costa, un río— lo hacemos con una
secuencia de vértices. Cuantos más vértices, más curvas capturamos y **más largo resulta**.
Si medimos con menos detalle, las curvas pequeñas desaparecen y el objeto se acorta.

Esto se conoce como la **paradoja de la costa**: la longitud de una costa depende de la
resolución con que se la mide, y no converge a un valor único. La escala no es un detalle
de presentación del mapa; **es parte de la definición de la medición**.

`simplify()` aplica el algoritmo de Douglas-Peucker: elimina los vértices que se apartan
menos que una tolerancia dada de la línea que los aproxima.

### Antes de medir: el sistema de coordenadas

Ojo con un detalle que viene de la Clase 1: para medir longitudes hay que estar en un
sistema de coordenadas **en metros**. Si midiéramos en grados, el número no significaría
nada.

In [ ]:
# Reproyectamos a un sistema que conserva distancias en Argentina
ruta40_m = ruta40.to_crs(EQUIDISTANTE)

chequear_crs(ruta40_m, proyectado=True, nombre="Ruta 40")
print("Ya podemos medir longitudes en metros")

### 👀 ¿Qué hace, exactamente, la tolerancia?

Acerquémonos a un tramo cualquiera de la ruta —una ventana de 40 × 40 km— y dibujemos
**cada vértice** como un punto negro. A la izquierda, la traza tal como viene en el archivo.
A la derecha, la misma traza después de simplificarla.

Simplificar es literalmente **tirar vértices**: la línea deja de seguir cada curva y pasa a
cortar camino en línea recta.

In [ ]:
# Elegimos un tramo cualquiera de la ruta y armamos una ventana de 40 km a su alrededor
coordenadas = ruta40_m.get_coordinates() # Obtenemos las coordenadas de los vertices de la linea. WKT Linestring(X1 Y1, X2, Y2,...)
tramo = coordenadas.iloc[20_000:20_400]  # Tomamos 400 pares de coordenadas. De la posición 20.000 a 20.400

# Creamos una ventana de visualización de 40 Km x 40 Km
centro_x = tramo["x"].mean()
centro_y = tramo["y"].mean()
LADO = 40_000

# Las tres versiones de la ruta que vamos a comparar
versiones = [
    ("Detalle completo", ruta40_m.geometry),
    ("Tolerancia 2 km",  ruta40_m.simplify(2_000)),
    ("Tolerancia 10 km", ruta40_m.simplify(10_000)),
]

fig, ejes = plt.subplots(1, 3, figsize=(13, 5))

for eje, (titulo, capa) in zip(ejes, versiones):
    vertices = capa.get_coordinates()
    capa.plot(ax=eje, color="#d95f02", linewidth=1.2)
    eje.scatter(vertices["x"], vertices["y"], s=12, color="black", zorder=3)

    eje.set_xlim(centro_x - LADO / 2, centro_x + LADO / 2)
    eje.set_ylim(centro_y - LADO / 2, centro_y + LADO / 2)
    eje.set_title(titulo)
    eje.set_xticks([])
    eje.set_yticks([])

plt.tight_layout()
plt.show()

### ▶️ Y ahora, cuánto mide cada versión

Cada vértice que se va se lleva un pedacito de curva, y con la curva se va longitud.
Midamos cuánta.

In [ ]:
# Longitud con todo el detalle que trae el archivo
longitud_original = ruta40_m.length.iloc[0] / 1000
vertices_original = len(ruta40_m.get_coordinates())

print(f"Detalle completo:   {longitud_original:>7,.0f} km   {vertices_original:>6} vértices")

# La misma ruta, dibujada con cada vez menos detalle
for tolerancia_km in [0.5, 2, 10, 50]:
    simplificada = ruta40_m.simplify(tolerancia_km * 1000)
    longitud = simplificada.length.iloc[0] / 1000
    vertices = len(simplificada.get_coordinates())
    perdida = 100 * (1 - longitud / longitud_original)
    print(f"Tolerancia {tolerancia_km:>4} km: {longitud:>7,.0f} km   {vertices:>6} vértices   "
          f"({perdida:.1f}% más corta)")

### ✅ Comprobación — ¿el número tiene sentido?

In [ ]:
LONGITUD_OFICIAL = 5194   # km declarados por Vialidad Nacional

diferencia = abs(longitud_original - LONGITUD_OFICIAL)

print(f"Longitud que medimos:  {longitud_original:,.0f} km")
print(f"Longitud oficial:      {LONGITUD_OFICIAL:,} km")
print(f"Diferencia:            {diferencia:,.0f} km ({100 * diferencia / LONGITUD_OFICIAL:.1f}%)")

### 🗺️ Las tres versiones, sobre el mapa entero

El zoom de recién mostraba 40 km. Así se ve la ruta completa, de La Quiaca a Cabo Vírgenes,
en los mismos tres niveles de detalle.

In [ ]:
# Preparamos las tres versiones de la ruta que vamos a comparar
capas = [
    ("Detalle completo", ruta40_m.geometry),
    ("Tolerancia 10 km", ruta40_m.simplify(10_000)),
    ("Tolerancia 50 km", ruta40_m.simplify(50_000)),
]

fig, ejes = plt.subplots(1, 3, figsize=(13, 7))

for eje, (titulo, capa) in zip(ejes, capas):
    capa.plot(ax=eje, color="#d95f02", linewidth=1)
    eje.set_title(f"{titulo}\n{capa.length.iloc[0] / 1000:,.0f} km · "
                  f"{len(capa.get_coordinates())} vértices")
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### 🔍 Interpretación

La Ruta 40 mide **5.166 km** con el máximo detalle disponible y **4.152 km** generalizada
a 50 km de tolerancia: un 19,6 % menos. Los tres mapas se ven casi iguales a esta escala
de impresión, pero responden distinto a la pregunta "¿cuánto mide?".

Fijate además que los primeros 41.000 vértices que eliminamos cuestan solo un 3,4 % de
longitud: la mayor parte del detalle no aporta información sustantiva, pero sí peso al
archivo.

> ⚠️ **Para tu investigación:** cuando compares magnitudes espaciales entre fuentes
> —longitud de red vial entre provincias, superficie de áreas protegidas entre países—
> asegurate de que provengan de cartografías de **escala comparable**. Una provincia no
> tiene más rutas que otra por estar mejor mapeada; pero el dato puede decir eso.

---

## 9. El problema de la unidad de área modificable (MAUP)

📽️ *Presentación Clase 2, diapositiva 11.*

### 🧭 Concepto

Muchas variables sociales **no se pueden medir en un punto**. La densidad de población, la
tasa de desempleo, el porcentaje de hogares con NBI: todas necesitan un área para existir.
Y las áreas que usamos —provincias, departamentos, radios censales, celdas de una grilla—
son **construcciones administrativas o analíticas**, no rasgos del territorio.

El problema de la unidad de área modificable, formulado por Openshaw en 1984, dice que
**los resultados dependen de esas áreas**. Y tiene dos componentes que conviene separar,
porque se corrigen de manera distinta:

| Componente | Qué cambia | Ejemplo |
|---|---|---|
| **Agregación** (o escala) | El **tamaño** de las unidades | Provincia vs. departamento vs. radio censal |
| **Zonificación** | La **forma o posición**, con el mismo tamaño | Dos grillas del mismo paso, corridas media celda |

Trabajamos sobre **Mendoza**, que tiene una estructura de poblamiento clara —un oasis
denso y un desierto vacío— donde el efecto se ve bien.

In [ ]:
mendoza          = provincias[provincias["provincia"] == "Mendoza"]
escuelas_mendoza = escuelas[escuelas["provincia"] == "Mendoza"]

superficie = mendoza.to_crs(EQUIVALENTE).area.iloc[0] / 1e6

print(f"Mendoza: {len(escuelas_mendoza)} escuelas en {superficie:,.0f} km²")
print(f"Densidad media: {len(escuelas_mendoza) / superficie * 100:.2f} escuelas cada 100 km²")

### 👀 La provincia y sus escuelas

Antes de dibujar ninguna grilla, miremos el dato crudo: dónde están efectivamente las 874
escuelas de Mendoza.

In [ ]:
fig, eje = plt.subplots(figsize=(6, 8))

mendoza.plot(ax=eje, color="#f0f0f0", edgecolor="black", linewidth=1)
escuelas_mendoza.plot(ax=eje, color="#d95f02", markersize=4)

eje.set_title(f"Mendoza: {len(escuelas_mendoza)} escuelas primarias")
eje.set_axis_off()
plt.show()

### ▶️ Efecto de zonificación

Tres grillas con **exactamente la misma superficie por celda** (625 km²). Lo único que
cambia es dónde caen las líneas divisorias.

In [ ]:
# Tres grillas con la misma superficie por celda: 625 km².
# Lo único que cambia es dónde caen las líneas divisorias.
cuadrados = grilla(mendoza, 25, "cuadrado")
corridos  = grilla(mendoza, 25, "cuadrado", desplazamiento=0.5)
hexagonos = grilla(mendoza, 25, "hexagono")

grillas = [
    ("Cuadrados", cuadrados),
    ("Cuadrados corridos media celda", corridos),
    ("Hexágonos de igual área", hexagonos),
]

for nombre, celdas in grillas:
    print(f"{nombre:32} {len(celdas):>4} celdas de {celdas['area_km2'].mean():>4.0f} km²")

### 👀 Las tres grillas sobre la provincia

Las mismas escuelas, con tres grillas de medición distintas. Mirá el oasis del norte, donde los
puntos se agrupan: **las líneas no caen en el mismo lugar**, y por lo tanto ese
agrupamiento no se reparte igual entre las celdas.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 6))

for eje, (nombre, celdas) in zip(ejes, grillas):
    celdas.boundary.plot(ax=eje, color="#3182bd", linewidth=0.4)
    mendoza.boundary.plot(ax=eje, color="black", linewidth=1)
    escuelas_mendoza.plot(ax=eje, color="#d95f02", markersize=2)

    eje.set_title(nombre)
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### ▶️ El conteo: cuántas escuelas caen en cada celda

Contamos las escuelas de cada celda y miramos dos cosas: la **media** y el **máximo**.

In [ ]:
def contar_escuelas(puntos, celdas):
    """Agrega a la grilla una columna con la cantidad de escuelas de cada celda."""
    celdas = celdas.copy()
    puntos_crs = puntos.to_crs(celdas.crs)
    union = gpd.sjoin(puntos_crs, celdas, predicate="within")
    conteo = union.groupby("celda_id").size()
    celdas["escuelas"] = celdas["celda_id"].map(conteo).fillna(0)
    return celdas


grillas_contadas = []

for nombre, celdas in grillas:
    celdas = contar_escuelas(escuelas_mendoza, celdas)
    grillas_contadas.append((nombre, celdas))

    print(f"{nombre:32} media {celdas['escuelas'].mean():>5.2f} · "
          f"MÁXIMO {celdas['escuelas'].max():>4.0f} escuelas en una celda")

### 🗺️ El mapa de conteos

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 6))

for eje, (nombre, celdas) in zip(ejes, grillas_contadas):
    celdas.plot(column="escuelas", cmap="OrRd", ax=eje,
                edgecolor="grey", linewidth=0.2,
                legend=True, legend_kwds={"shrink": 0.5})
    mendoza.boundary.plot(ax=eje, color="black", linewidth=0.8)

    eje.set_title(f"{nombre}\nmáximo = {celdas['escuelas'].max():.0f}")
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### 🔍 Interpretación

Las tres grillas tienen la misma superficie por celda, casi la misma cantidad de celdas y
prácticamente la misma media: **3,1 escuelas por celda** en los tres casos. Pero el
**máximo de escuelas en una celda** pasa de **283 a 167** con solo correr la grilla media
celda: una caída del **41 %**. Cambiar cuadrados por hexágonos de igual área, en cambio,
casi no lo mueve (287).

Ese máximo es lo que en un informe se llamaría *"la zona de mayor concentración educativa
de Mendoza"*. Y su valor depende de dónde dibujamos las líneas.

> La media es robusta a la zonificación; los **extremos no lo son**.
> Prestar atención cuando en los informes aparece: el barrio más carenciado, la zona más
> insegura, el área con menor cobertura.

### ▶️ Efecto de agregación

La misma forma pero distinto tamaño.

### 👀 La misma provincia, con celdas cada vez más grandes

Cuatro grillas de cuadrados: 10, 25, 50 y 100 km de lado. Como las celdas tienen tamaños
distintos, no podemos comparar conteos; comparamos **densidad**, escuelas cada 100 km².

> ⚠️ **ATENCION**: Cada panel tiene su **propia escala de color**: el valor máximo está escrito en el
> título. Mirá cómo la mancha roja se agranda mientras el número que la acompaña se reduce.

In [ ]:
tamanos = [10, 25, 50, 100]
grillas_por_tamano = []

fig, ejes = plt.subplots(1, 4, figsize=(16, 5))

for eje, lado_km in zip(ejes, tamanos):
    celdas = contar_escuelas(escuelas_mendoza, grilla(mendoza, lado_km, "cuadrado"))
    celdas["densidad"] = celdas["escuelas"] / celdas["area_km2"] * 100
    grillas_por_tamano.append((lado_km, celdas))

    celdas.plot(column="densidad", cmap="OrRd", ax=eje, edgecolor="grey", linewidth=0.1)
    mendoza.boundary.plot(ax=eje, color="black", linewidth=0.8)

    eje.set_title(f"Celdas de {lado_km} km\ndensidad máx = {celdas['densidad'].max():.0f}")
    eje.set_axis_off()

plt.tight_layout()
plt.show()

### ▶️ Los números detrás de esos cuatro mapas

El **coeficiente de variación** resume cuán desparejo es el mapa: cuanto más alto, más
diferencias hay entre celdas.

In [ ]:
for lado_km, celdas in grillas_por_tamano:
    densidad = celdas["densidad"]
    variacion = densidad.std() / densidad.mean()

    print(f"Celdas de {lado_km:>3} km de lado: {len(celdas):>4} celdas · "
          f"máx {celdas['escuelas'].max():>4.0f} escuelas · "
          f"densidad máx {densidad.max():>6.1f} · "
          f"coef. de variación {variacion:.1f}")

### 🔍 Interpretación

Al agrandar las celdas pasan dos cosas opuestas y ambas importan:

- El **conteo máximo por celda sube** (de 149 a 515): celdas más grandes contienen más cosas.
- La **densidad máxima baja** (de 149 a 5,1 escuelas cada 100 km²) y el **coeficiente de
  variación cae** (de 8,4 a 3,1): al agregar, promediamos zonas densas con zonas vacías y
  la variabilidad se suaviza.

Esto último es lo grave: **agregar oculta desigualdad**. Un mapa por provincia de un
indicador social siempre se verá más homogéneo que el mismo indicador por radio censal,
aunque la desigualdad subyacente sea idéntica. La elección de la escala determina cuánta
desigualdad es visible.

> ⚠️ **No hay una escala "correcta".** Hay una escala **adecuada al fenómeno**. Si estudiás
> accesibilidad a una escuela primaria, la unidad razonable es el radio censal o el barrio,
> porque nadie recorre 100 km para ir a la escuela todos los días. Si estudiás política educativa
> provincial, la provincia tiene sentido porque es la unidad donde se decide.

---

## 10. Efecto de borde

📽️ *Presentación Clase 2, diapositiva 14.*

### 🧭 Concepto

Todo análisis espacial se hace sobre una **zona de estudio** con un límite. Y ese límite
casi nunca es un límite del fenómeno: es un límite de nuestros datos, o de la jurisdicción
que los publica.

Las unidades que caen cerca del borde tienen vecinos **del otro lado** que no estamos
contando. Cualquier medida que dependa del entorno —densidad, cantidad de vecinos,
promedio de la zona— queda **subestimada** ahí, y esa subestimación no es ruido aleatorio:
es un sesgo sistemático que apunta siempre en la misma dirección.

El caso argentino más claro es la **Ciudad de Buenos Aires**. Es una jurisdicción con datos
propios, presupuesto propio y sistema educativo propio, y por eso casi todos los estudios
la toman como zona de estudio. Pero del otro lado de la General Paz y del Riachuelo el
tejido urbano sigue sin interrupción, y miles de chicos cruzan ese límite todos los días
para ir a la escuela.

Vamos a medirlo: para cada escuela de la Ciudad contamos cuántas escuelas hay **a menos de
1 km**, primero mirando solo las de la Ciudad y después agregando las del resto del país.

In [ ]:
escuelas_m = escuelas.to_crs(EQUIVALENTE)

# La zona de estudio: el polígono de la Ciudad, que ya viene en la capa de provincias
caba   = provincias[provincias["provincia"] == "Ciudad Autónoma de Buenos Aires"].to_crs(EQUIVALENTE)
ciudad = caba.geometry.iloc[0]

RADIO = 1_000   # el entorno que vamos a mirar alrededor de cada escuela

dentro = escuelas_m[escuelas_m.within(ciudad)].copy()
fuera  = escuelas_m[~escuelas_m.within(ciudad)]

# Cuáles están cerca del límite: a esas les va a faltar parte del entorno
dentro["distancia_al_borde"] = dentro.distance(ciudad.boundary)
dentro["en_el_borde"] = dentro["distancia_al_borde"] < RADIO

# Y cuántas escuelas de afuera están lo bastante cerca como para ser vecinas de alguna
fuera_cerca = fuera[fuera.distance(ciudad.boundary) < RADIO]

print(f"Escuelas dentro de la Ciudad:                {len(dentro)}")
print(f"De ellas, a menos de 1 km del límite:        {dentro['en_el_borde'].sum()}")
print(f"Escuelas de afuera, a menos de 1 km:         {len(fuera_cerca)}")
print(f"Escuelas en todo el país:                    {len(escuelas_m)}")

### 👀 La Ciudad, su límite y lo que hay del otro lado

En gris el interior, en naranja las escuelas que están a menos de 1 km del límite, y en
azul **las que quedan afuera de la zona de estudio**. Mirá el azul: no hay ningún vacío del
otro lado de la General Paz ni del Riachuelo. La ciudad se termina en el mapa, no en el
territorio.

In [ ]:
minx, miny, maxx, maxy = caba.total_bounds
MARGEN = 4_000   # cuánto mostramos alrededor de la Ciudad

fig, eje = plt.subplots(figsize=(9, 9))

caba.plot(ax=eje, color="#f0f0f0", edgecolor="black", linewidth=1.5)
fuera.plot(ax=eje, color="#4292c6", markersize=5,
           label="Fuera de la Ciudad (Gran Buenos Aires)")
dentro[~dentro["en_el_borde"]].plot(ax=eje, color="#969696", markersize=5,
                                    label="Adentro, interior")
dentro[dentro["en_el_borde"]].plot(ax=eje, color="#d95f02", markersize=7,
                                   label="Adentro, a menos de 1 km del límite")

eje.set_xlim(minx - MARGEN, maxx + MARGEN)
eje.set_ylim(miny - MARGEN, maxy + MARGEN)
eje.set_title("La Ciudad de Buenos Aires como zona de estudio\n"
              "y las escuelas que quedan del otro lado del límite")
eje.legend(loc="upper left")
eje.set_axis_off()
plt.show()

### 👀 Una escuela del borde, de cerca

Tomemos la escuela más pegada al límite de todas y dibujemos su entorno de 1 km. Es el
mismo círculo que vamos a usar para contar vecinas, pero de a una se ve lo que el promedio
esconde.

In [ ]:
escuela = dentro.sort_values("distancia_al_borde").iloc[0] # ordeno por distancia al borde y obtengo la primera (menor distancia)
circulo = escuela.geometry.buffer(RADIO) # Simulo un radio (buffer) de 1Km

en_circulo_dentro = dentro[dentro.within(circulo)] # Obtengo las escuelas dentro de la ciudad y del radio
en_circulo_fuera  = fuera[fuera.within(circulo)] # Obtengo las escuelas fuera de la ciudad y dentro del radio

fig, eje = plt.subplots(figsize=(7, 7))

caba.plot(ax=eje, color="#f0f0f0", edgecolor="black", linewidth=1.5)
gpd.GeoSeries([circulo], crs=EQUIVALENTE).plot(ax=eje, facecolor="none",
                                               edgecolor="#d95f02", linewidth=1.5)
en_circulo_dentro.plot(ax=eje, color="#969696", markersize=30, label="Vecinas que SÍ contamos")
en_circulo_fuera.plot(ax=eje, color="#4292c6", markersize=30, label="Vecinas que NO contamos")
gpd.GeoSeries([escuela.geometry], crs=EQUIVALENTE).plot(ax=eje, color="#d95f02",
                                                        markersize=110, label="La escuela")

# Dos carteles, para que se vea de qué lado está cada cosa
eje.text(escuela.geometry.x + 1350, escuela.geometry.y + 1150, "Ciudad de\nBuenos Aires",
         ha="right", fontsize=11, color="#525252")
eje.text(escuela.geometry.x - 1400, escuela.geometry.y - 1250, "Gran Buenos\nAires",
         ha="left", fontsize=11, color="#4292c6")

eje.set_xlim(escuela.geometry.x - 1500, escuela.geometry.x + 1500)
eje.set_ylim(escuela.geometry.y - 1500, escuela.geometry.y + 1500)
eje.set_title(f"{escuela['establecimiento']}\n"
              "y su entorno de 1 km, partido por el límite de la Ciudad")
eje.legend(loc="upper left", fontsize=9)
eje.set_axis_off()
plt.show()

print(f"Vecinas dentro de la Ciudad: {len(en_circulo_dentro) - 1}")
print(f"Vecinas del otro lado:       {len(en_circulo_fuera)}")

### ▶️ Contamos las vecinas de cada escuela, de dos maneras

Lo que acabamos de mirar en una escuela, ahora en las 894. Alrededor de cada una dibujamos
el círculo de 1 km y contamos cuántas escuelas caen adentro. Primero contando **solo las de
la Ciudad**, que es lo que haría cualquiera que trabaja con datos de una jurisdicción.
Después contando **todas las del país**, que es la respuesta correcta y que acá podemos
calcular porque tenemos el padrón completo.

In [ ]:
# Un círculo de 1 km alrededor de cada escuela de la Ciudad
circulos = dentro[["geometry"]].copy()
circulos["geometry"] = dentro.buffer(RADIO)

# Cuántas escuelas caen dentro de cada círculo, contando SOLO las de la Ciudad
union_caba = gpd.sjoin(circulos, dentro[["geometry"]], predicate="contains")
dentro["vecinas_caba"] = union_caba.index.value_counts() - 1   # no se cuenta a sí misma

# Lo mismo, pero contando TODAS las escuelas del país
union_pais = gpd.sjoin(circulos, escuelas_m[["geometry"]], predicate="contains")
dentro["vecinas_pais"] = union_pais.index.value_counts() - 1

# Cuántas vecinas se pierde cada escuela por mirar solo la Ciudad
dentro["faltan"] = dentro["vecinas_pais"] - dentro["vecinas_caba"]

dentro[["establecimiento", "vecinas_caba", "vecinas_pais", "faltan"]].head()

### ✅ Comprobación
Veamos que ocurre con las escuelas que se encuentran "en el borde" y las que se encuentran en el "interior".

In [ ]:
comparacion = dentro.groupby("en_el_borde")[["vecinas_caba", "vecinas_pais"]].mean()
comparacion["subestimacion_%"] = 100 * (1 - comparacion["vecinas_caba"] / comparacion["vecinas_pais"])
comparacion.index = ["Interior", "Franja de borde"]

comparacion.round(1)

### ✅ Comprobación — ¿a cuántas escuelas les pasa de verdad?

El promedio de la franja mezcla dos situaciones muy distintas, y conviene separarlas: hay
escuelas del borde que no pierden ni una vecina.

In [ ]:
franja    = dentro[dentro["en_el_borde"]] # Nos quedamos solamente con las que estan en la franja (color naranja)
afectadas = franja[franja["faltan"] > 0]  # Y con las afectadas

print(f"Escuelas en la franja de borde:            {len(franja)}")
print(f"De ellas, pierden al menos una vecina:     {len(afectadas)}")
print(f"Y no pierden ninguna:                      {len(franja) - len(afectadas)}")

media_caba = afectadas["vecinas_caba"].mean()
media_pais = afectadas["vecinas_pais"].mean()
print(f"\nEn las afectadas: {media_caba:.1f} vecinas contadas contra {media_pais:.1f} reales "
      f"-> subestimación del {100 * (1 - media_caba / media_pais):.0f}%")

# La más golpeada en proporción: es la misma que dibujamos recién
afectadas = afectadas.copy()
afectadas["porcentaje_perdido"] = 100 * afectadas["faltan"] / afectadas["vecinas_pais"]
peor = afectadas.sort_values("porcentaje_perdido", ascending=False).iloc[0]
print(f"Peor caso: {peor['establecimiento']}, "
      f"{peor['vecinas_caba']} contadas contra {peor['vecinas_pais']} reales "
      f"({peor['porcentaje_perdido']:.0f}% de su entorno)")

### ✅ Comprobación — ¿y si el entorno fuera más grande?

Un kilómetro es un entorno chico. Muchos análisis usan radios mayores: áreas de influencia,
promedios móviles, matrices de pesos por distancia. Cuanto más lejos se mira, más ancha es
la franja contaminada.

In [ ]:
for radio_km in [1, 2, 5]:
    en_franja = dentro["distancia_al_borde"] < radio_km * 1000
    print(f"Entorno de {radio_km} km: {en_franja.sum():>4} escuelas en la franja "
          f"({100 * en_franja.mean():.0f}% de las escuelas de la Ciudad)")

### 🔍 Interpretación

En el **interior la subestimación es exactamente 0 %**: 18,0 vecinas contando solo la
Ciudad y 18,0 contando todo el país.

En la **franja de borde** el promedio da 10,0 contra 10,7: un 7 %. Parece poco, y ahí está
lo interesante: **ese promedio esconde dos situaciones opuestas**. De las 138 escuelas de la
franja, 87 no pierden ni una sola vecina, y 51 pierden en serio. Entre estas últimas la
subestimación sube al **17 %**, y la escuela más pegada al límite —Soldado de Malvinas—
cuenta **5 vecinas cuando en realidad tiene 9**: se pierde el 44 % de su entorno.

La razón se ve en el primer mapa. El límite de la Ciudad es de dos clases distintas:

- Al **este** es el Río de la Plata. Ahí no hay nada del otro lado, y no lo hay de verdad:
  el límite de los datos coincide con un límite del fenómeno. Esas escuelas están en la
  franja de borde y no sufren ningún sesgo.
- Al **oeste y al sur** son la General Paz y el Riachuelo. Ahí el tejido urbano sigue
  igual, hay 83 escuelas a menos de 1 km del otro lado, y el sesgo es fuerte.

> 🔍 **La regla, entonces, no es "cuidado con los bordes".** Es: *cuidado con los bordes
> que son límites de tus datos y no del fenómeno que estudiás.* Un río, una cordillera o el
> mar cortan de verdad. Una avenida que separa dos jurisdicciones, no.

Un detalle metodológico que conviene no confundir: las escuelas de la franja tienen menos
vecinas que las del interior (10,7 contra 18,0) **aun contando todo el país**. Eso no es
efecto de borde, es que el borde de esta ciudad en particular tiene el puerto, la reserva
ecológica y las vías: es realmente menos denso. El efecto de borde es la comparación
**dentro** de la franja, 10,0 contra 10,7.

Y una advertencia de escala. Con un entorno de 1 km la franja alcanza al 15 % de las
escuelas; con 2 km, al 36 %; con 5 km, al **90 %**. La Ciudad mide unos 20 por 10 km, así
que apenas el análisis mira un poco más lejos, "el borde" deja de ser un margen y pasa a ser
casi toda la zona de estudio. En zonas chicas el efecto de borde no es un detalle a declarar
al final: condiciona si el análisis se puede hacer.

> ⚠️ **Qué hacer.** Tres estrategias, en orden de preferencia: (1) traer datos de una zona
> más amplia que la de análisis y usar ese margen solo para el cálculo —acá sería trabajar
> con el AMBA aunque el informe hable de la Ciudad—; (2) restringir las conclusiones al
> interior; (3) si no se puede ninguna de las dos, **declarar el problema** y no interpretar
> los valores del borde. Lo que nunca corresponde es presentar el borde como si fuera
> comparable con el interior.

---

## 11. Localización representada

📽️ *Presentación Clase 2, diapositiva 15.*

### 🧭 Concepto

Cuando un dato está agregado por área, tarde o temprano necesitamos **un punto** que la
represente: para calcular distancias, para dibujar un símbolo proporcional, para medir
vecindad. La elección habitual es el **centroide geométrico**, el centro de masa del
polígono suponiendo densidad uniforme.

Ese supuesto —densidad uniforme— casi nunca se cumple en variables sociales. La gente no
está repartida de manera pareja dentro de una provincia.

Comparamos el centroide geométrico de cada provincia contra el **centro de sus escuelas**,
que funciona como una aproximación a dónde está realmente la población.

### 👀 Los dos centros de Buenos Aires

El punto negro es el centroide geométrico: el centro de masa del polígono, suponiendo que
la provincia está rellena de manera pareja. 
El punto naranja es el promedio de las
posiciones de sus escuelas, que se parece bastante más a dónde vive la gente.

In [ ]:
buenos_aires = provincias[provincias["provincia"] == "Buenos Aires"].to_crs(EQUIVALENTE)
escuelas_ba  = escuelas[escuelas["provincia"] == "Buenos Aires"].to_crs(EQUIVALENTE)

centro_geometrico = buenos_aires.geometry.centroid.iloc[0] # Selecciona la geometría (poligono de la provincia y calculo su centroide)
centro_escuelas   = Point(escuelas_ba.geometry.x.mean(), escuelas_ba.geometry.y.mean()) # Tomo todas las escuelas y calculo el promedio de sus coordeandas X e y. Construyo un punto con ello.

fig, eje = plt.subplots(figsize=(7, 8))
buenos_aires.plot(ax=eje, color="#f0f0f0", edgecolor="grey", linewidth=0.6)
escuelas_ba.plot(ax=eje, color="#bdbdbd", markersize=1)

eje.scatter(centro_geometrico.x, centro_geometrico.y, s=120, color="black",
            label="Centroide geométrico")
eje.scatter(centro_escuelas.x, centro_escuelas.y, s=120, color="#d95f02",
            label="Centro de las escuelas")

eje.set_title("Provincia de Buenos Aires: dos puntos que la representan")
eje.legend()
eje.set_axis_off()
plt.show()

### ▶️ ¿Y en otras provincias?

La distancia entre los dos puntos es una medida de cuánto miente el centroide geométrico.
Veamos cuánto vale en cinco provincias con formas y asentamientos humanos muy distintos.

In [ ]:
provincias_a_mirar = ["Buenos Aires", "Mendoza", "Santa Cruz", "Chubut", "Salta"]

for nombre in provincias_a_mirar:
    poligono = provincias[provincias["provincia"] == nombre].to_crs(EQUIVALENTE)
    puntos   = escuelas[escuelas["provincia"] == nombre].to_crs(EQUIVALENTE)

    centro_geometrico = poligono.geometry.centroid.iloc[0]
    centro_escuelas   = Point(puntos.geometry.x.mean(), puntos.geometry.y.mean())

    distancia_km = centro_geometrico.distance(centro_escuelas) / 1000

    print(f"{nombre:15} {len(puntos):>5} escuelas · "
          f"{distancia_km:>6.1f} km entre los dos centros")

### 🔍 Interpretación

En Buenos Aires los dos puntos están a **186 km** de distancia. El mapa lo muestra sin
ambigüedad: el centroide geométrico cae en el centro de la provincia, en una zona con
pocas escuelas, mientras que el centro de las escuelas se corre hacia el nordeste, hacia el
conurbano, que es donde vive la gente.

Si usáramos el centroide geométrico para calcular, por ejemplo, "la distancia media de un
bonaerense a un hospital de alta complejidad", el resultado estaría equivocado por cientos
de kilómetros, y equivocado **en contra** de la población más numerosa.

Fijate que el error no es parejo: en Salta son 20 km y en Buenos Aires 186.

> ⚠️ Por eso las alternativas al centroide geométrico —el centroide ponderado por población,
> o un punto interior representativo— importan tanto en Argentina, donde la concentración
> urbana es extrema. Volvemos sobre esto en las clases 6 y 8.

---

## 12. Cierre

### Glosario de la clase

| Término | Definición |
|---|---|
| **Shapefile** | Formato vectorial repartido en cuatro archivos como mínimo; corta los nombres de campo a diez caracteres. |
| **GeoPackage** | Formato vectorial abierto de un solo archivo, sin límite práctico en los nombres de campo. Es el que usamos para guardar. |
| **Primera ley de la geografía** | Las cosas cercanas se parecen más entre sí que las lejanas (Tobler, 1970). |
| **Autocorrelación espacial** | Que el valor de una variable en un lugar dependa de su valor en los lugares vecinos. Viola el supuesto de independencia. |
| **Escala de análisis** | Nivel de detalle de la medición. Determina qué se puede observar y cuánto mide lo que se mide. |
| **Generalización** | Simplificación de una geometría reduciendo vértices. |
| **MAUP** | Que los resultados dependan de las unidades areales elegidas, que son arbitrarias. |
| **Efecto de agregación** | Componente del MAUP asociado al **tamaño** de las unidades. |
| **Efecto de zonificación** | Componente del MAUP asociado a la **forma o posición** de las unidades. |
| **Efecto de borde** | Sesgo en los cálculos de las unidades cercanas al límite de la zona de estudio. |
| **Localización representada** | El punto que se elige para representar un área; el centroide geométrico supone densidad uniforme. |

---

### Lo que quedó pendiente

- Hoy usamos dos **sistemas de coordenadas**, también dijimos que medir en grados no sirve. La **Clase 4** se ocupa de esto.
- Vimos de manera intuitiva el concepto de **autocorrelación espacial**. El índice de Moran y los métodos formales son la **Clase 7**.
- Trabajamos con **operaciones espaciales** simples: uniones espaciales, calculo de centroides, buffers, etc. Retomaremos estas operaciones en la **Clase 6**.

## 13. Bibliografía

- Olaya, V. (2020). *Sistemas de Información Geográfica*, cap. "Conceptos básicos para el
  análisis espacial". Edición libre, CC BY. <https://volaya.github.io/libro-sig/>
- Openshaw, S. (1984). *The Modifiable Areal Unit Problem*. Concepts and Techniques in
  Modern Geography (CATMOG) 38. Norwich: Geo Books.
- Tobler, W. R. (1970). "A Computer Movie Simulating Urban Growth in the Detroit Region".
  *Economic Geography*, 46(sup1), 234–240. <https://doi.org/10.2307/143141>
